In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.autolayout"] = True # to for tight_layout()
import pymc as pm
import xarray as xr
import scipy as sp
from scipy.special import expit as logistic
from scipy.stats import gamma
import pytensor.tensor as tt

# Mixtures (Generalized GLMs)

* More realistic, try to model reality (instead of transforming data to fit the model)

<img style="width: 40%; float: right" src="figs/mixing_chemicals.jpg">

## Agenda

* Over-dispersion
    * Beta-Binomial
    * Gamma-Poisson
* Zero inflated outcomes
    * Zero-Inflated Poisson
* Ordinal Regression 
    * Ordinal outcome (predicted variable)
    * Ordinal predictor


## Over-dispersion


* Remember that Gaussian models were very sensitive to outliers    
    * The inferred variance of the distributions was large    
    * The Student-t distribution helps accommodating for outliers

<br>

* Count models also may also be sensitive to outliers

<br>

* When data is more variable than what a pure model assumes, then we say that it exhibits __over-dispersion__

<br>

* Often over-dispersion appears due to *missing predictors*
    * Ignoring over-dispersion may lead to spurious inferences  
    * The mixture models we will discuss may help to capture over-dispersion 
    * However, when possible, it is better to include any missing predictors causing the over-dispersion

### Beta-Binomial

#### Beta distribution (revisited)

* The *Beta* distribution is a continuous distribution whose support is (0,1)

<br>

* The Beta distribution has two shape parameters $\alpha$ and $\beta$, $\mathrm{Beta}(\alpha,\beta)$
    $$
    p(\theta \mid \alpha,\beta) = \frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}{B(\alpha,\beta)}
    $$
  where $B(\alpha,\beta) = \int^1_0 \theta^{\alpha-1}(1-\theta)^{\beta-1} d\theta$ is the normalizing factor

<br>

* These parameters define the shape of the distribution

* The parameters $\alpha$ and $\beta$ do not have a very intuitive interpretation

<br>

* The book uses an alternative way to define the shape of the distribution in terms of:
    * The average probability ($\bar{p}$) and,
    * shape ($\theta$) or concentration (when $\theta = 2$ means equal probabilities, $\theta > 2$ greater concentration around $\bar{p}$ and $\theta<2$ more dispersion)

<br>

* Given the above parameters, we can define $\mathrm{Beta}(\alpha = \bar{p}\theta, \beta=(1-\bar{p})\theta)$



In [ ]:
x_plot = np.linspace(0, 1, 100)
pbars  = [0.5,0.5,0.5,0.5,0.2,0.2,0.2,0.2]
thetas = [10.0,5.0,2.0,1.0,10.0,5.0,2.0,1.0]
fig, axs = plt.subplots(2, 4,figsize=(10,5))
for pbar,theta,ax in zip(pbars,thetas,axs.ravel()):
    ax.plot(
        x_plot,
        sp.stats.beta.pdf(x_plot[:, np.newaxis], pbar * theta, (1 - pbar) * theta)
    )
    ax.vlines(x=pbar,ymin=0,ymax=3.5,linestyle='--',color='black',lw=.5,label='$\\bar{p}$')
    ax.set_title(label=f"$\\bar p = {pbar}, \\theta= {theta}$\n$Beta(\\alpha={pbar * theta},\\beta={(1 - pbar) * theta})$")
    ax.legend();

#### Beta-Binomial Model

* Key: It is a **mixture** of binomial distributions

<br>


* **This model assumes a different $p$ for each row in the dataset**
    * That is, each row has its own unobserved success rate
    * We do not estimate a distribution of probabilities of success, but **a distribution of distributions of probability of success**
    
<br>

* The model makes use of the conjugacy property of Beta priors and Binomial likelihoods

<br>

* Recall the UC Berkely admissions model, data was dispersed if we ignore the predictor "department"

In [ ]:
d_ucb = pd.read_csv("Data/UCBadmit.csv", sep=";")
N = d_ucb.applications.values
gid = (d_ucb["applicant.gender"] == "female").astype(int).values
d_ucb.head()

* We apply a Beta-Binomial model with a categorical predictor (`gid`)

\begin{align*}
A_i \sim & \; \mathrm{BetaBinomial}(N_i,\bar{p}_i,\theta) \\
\mathrm{logit}(\bar{p}_i) = &\; \alpha_{\mathrm{GID}[o]} \\
\alpha_j \sim &\; \mathrm{Normal}(0,1.5) \\
\theta = &\; \varphi + 2 \\
\varphi = &\; \mathrm{Exponential}(1)
\end{align*}

* We define $\theta = \varphi + 2$, as $\varphi > 2$ represents a more concentrated distribution

In [ ]:
with pm.Model() as m12_1:
    φ    = pm.Exponential('φ', lam=1)
    θ    = pm.Deterministic('θ', φ + 2) # so that it is part of the trace
    α    = pm.Normal('α',mu=0,sigma=1.5,shape=np.unique(gid).size)
    pbar = pm.Deterministic('pbar', pm.invlogit(α[gid])) # so that it is part of the trace
    A    = pm.BetaBinomial('A',n=N,alpha=pbar*θ,beta=(1-pbar)*θ,observed=d_ucb.admit.values)
    
    # difference of log-odds and probability scale
    pm.Deterministic('diff_α', α[0]-α[1])

In [ ]:
pm.model_to_graphviz(m12_1)

In [ ]:
trace_m12_1 = pm.sample(model=m12_1,return_inferencedata=True)

In [ ]:
pm.summary(trace_m12_1,var_names=['α','φ','θ','diff_α'])

* Recall that the $\alpha$ parameters are in log-odds

<br>

* Note that the contrast $\text{diff}\_\alpha$ is negative, but also also quite widespread. Meaning that there is not evidence for a bias in the admission rates
    * The following posterior plot visualizes this 

In [ ]:
pm.plot_posterior(trace_m12_1, var_names=['diff_α'], ref_val=0.0, hdi_prob=0.95);

* It is important to note that, in this Beta-Binomial model, the $\bar{p}$ and $\theta$ samples give us a **distribution of Beta posterior distributions** 
    * Note the difference with the regular Binomial regression model that gives a distribution over values of $p$

<br>

<div style="width:700px;  height: 40px; display:flex; align-items:center; justify-content:center; background:blue; text-align:center; box-sizing:border-box; margin:8px auto; color: white">
    <strong style="font-size: 16px">
        Why do we know that the posterior samples of $\bar{p}$ and $\theta$ define Beta distributions?
    </strong>
</div>

<br>

* The plot below shows that there is a lot of variation
    * Overall all $\bar{p}$ values are plausible

In [ ]:
gid = 1
x_plot = np.linspace(0, 1, 100)

trace_pbar_gid = trace_m12_1.posterior.sel(α_dim_0=gid)['pbar'].values.ravel()
trace_θ_gid = trace_m12_1.posterior.sel(α_dim_0=gid)['θ'].values.ravel()

trace_pbar_gid_mean = trace_pbar_gid.mean()
trace_θ_gid_mean = trace_θ_gid.mean()

# mean Beta distribution
plt.plot(x_plot,sp.stats.beta.pdf(x_plot,trace_pbar_gid_mean*trace_θ_gid_mean,
                                  (1-trace_pbar_gid_mean)*trace_θ_gid_mean),color='k');

# 50 sampled Beta posterior distributions
n_samples = 50
for pbar,theta in zip(trace_pbar_gid[-n_samples:],trace_θ_gid[-n_samples:]):
    plt.plot(x_plot,sp.stats.beta.pdf(x_plot,pbar*theta,(1-pbar)*theta),alpha=.1,color='k');
plt.title(f'Posterior distribution of Beta distributions for female (gid={gid})');
plt.ylim(0.0, 3.0)
plt.show();

* Finally, we perform a posterior predictive check to assess whether the model fits the data

* Although it does, we observe that the model produces notably predictions

In [ ]:
with m12_1:
    ppc_test = pm.sample_posterior_predictive(trace_m12_1)

In [ ]:
with m12_1:
    ppc = pm.sample_posterior_predictive(trace_m12_1).posterior_predictive["A"]
pp_admit = ppc / N

In [ ]:
# adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_12.ipynb
trace_m12_1_pbar = np.concatenate(trace_m12_1.posterior["pbar"],axis=0)
plt.plot(range(1, 13), d_ucb.admit / N, "C0o", ms=6, alpha=0.6)
plt.plot(range(1, 13), trace_m12_1_pbar.mean(0), "ko", fillstyle="none", ms=6, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], pm.hdi(trace_m12_1_pbar[None, :]).T, "k-", lw=1, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], pm.hdi(pp_admit)['A'].T, "k+", ms=6, alpha=0.6);

### Gamma-Poisson

* The Gamma distribution is a generalization of the Exponential distribution
    * The Gamma distribution is a conjugate prior for Poisson models
    * As in Beta-Binomial, the Gamma distribution is chosen due to mathematical convenience    
<br>

* The Gamma-Poisson model assumes a different rate parameter ($\lambda$) for each row in the dataset.

<br>

* The Gamma-Poisson distribution has two parameters, rate ($\lambda$) and dispersion or scale ($\varphi$)

$$
\mathrm{Gamma\textrm{-}Poisson}(\lambda,\varphi)
$$

<br>


    
#### Gamma-Poisson Model

* We revisit the oceanic islands and tools (scientific) model

* Now we use a Gamma-Poisson model to account for over-dispersion

* The model is defined as

\begin{align*}
T_i \sim &\; \mathrm{Gamma\textrm{-}Poisson}(\lambda_i,\varphi)\\
\lambda_i = &\; \frac{\alpha_{\mathrm{CID}[i]}P_i^{\beta_{\mathrm{CID}[i]}}}{\gamma} \\
\alpha_i \sim &\; \mathrm{Normal}(1,1)\\
\beta_i \sim &\; \mathrm{Exponential}(1)\\
\gamma \sim &\; \mathrm{Exponential}(1)\\
\varphi \sim &\; \mathrm{Exponential}(1)
\end{align*}

In [ ]:
d_k = pd.read_csv("Data/Kline.csv", delimiter=";")
c_id = (d_k.contact == "high").astype(int).values
d_k

In [ ]:
# from previous lecture
with pm.Model() as m11_11:
        γ   = pm.Exponential('γ',1)
        β   = pm.Exponential('β',1,shape=np.unique(c_id).size)
        α   = pm.LogitNormal('α',1,1,shape=np.unique(c_id).size)
        cid = pm.Data("cid", c_id)
        P   = pm.Data("P", d_k.population)
        λ   = pm.Deterministic('λ', α[cid]*tt.math.pow(P,β[cid])/γ)
        T   = pm.Poisson('T',mu=λ,observed=d_k.total_tools)
        trace_m11_11 = pm.sample(idata_kwargs={'log_likelihood': True})

# Gamma-Poisson models
with pm.Model() as m12_2:
        γ   = pm.Exponential('γ',1)
        β   = pm.Exponential('β',1,shape=np.unique(c_id).size)
        α   = pm.LogitNormal('α',1,1,shape=np.unique(c_id).size)
        cid = pm.Data("cid", c_id)
        P   = pm.Data("P", d_k.population)
        λ   = pm.Deterministic('λ', α[cid]*tt.math.pow(P,β[cid])/γ)
        
        # only new changes to turn it in to a Gamma-Poisson
        φ   = pm.Exponential('φ',1)
        T   = pm.NegativeBinomial('T',λ,φ,observed=d_k.total_tools)
        trace_m12_2 = pm.sample(idata_kwargs={'log_likelihood': True})

* The plot below shows the posterior distribution over Gamma distributions for the Poisson rate for island 0
    * Minor detail, the `scipy` library uses shape (`a`) and scale to define the gamma distribution. The scale is defined as the mean of the distribution divided by its shape.

In [ ]:
island_id = 0
samples = 50
means = trace_m12_2.posterior.λ.sel(λ_dim_0=island_id).values.ravel()[-samples:]
alphas = trace_m12_2.posterior.φ.values.ravel()[-samples:]
x = np.linspace(0,100, num=200)

fig, ax = plt.subplots()

for a, m in zip(alphas, means):
    ax.plot(x, gamma(a=a,scale=m/a).pdf(x), color='k', alpha=0.1)
ax.plot(x, gamma(a=alphas.mean(),scale=(means/alphas).mean()).pdf(x), color='k', alpha=1)
plt.title(f'Posterior distribution of Gamma distributions for island {island_id}');

* Here we compare the posterior predictive distribution for the regular Poisson model and the mixture $\textrm{Gamma-Poisson}$

In [ ]:
# adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_12.ipynb
ns = 10
P_seq = d_k.population

with m11_11:
    # predictions for cid=0 (low contact)
    pm.set_data({"cid": np.array([0] * ns), "P": P_seq})
    lam0_11 = pm.sample_posterior_predictive(trace_m11_11, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

    # predictions for cid=1 (high contact)
    pm.set_data({"cid": np.array([1] * ns)})
    lam1_11 = pm.sample_posterior_predictive(trace_m11_11, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

with m12_2:
    pm.set_data({"cid": np.array([0] * ns), "P": P_seq})
    lam0_12 = pm.sample_posterior_predictive(trace_m12_2, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

    pm.set_data({"cid": np.array([1] * ns)})
    lam1_12 = pm.sample_posterior_predictive(trace_m12_2, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

lmu0_11, lmu1_11 = lam0_11.mean(["chain", "draw"]), lam1_11.mean(["chain", "draw"])
lmu0_12, lmu1_12 = lam0_12.mean(["chain", "draw"]), lam1_12.mean(["chain", "draw"])

In [ ]:
# scale point size to Pareto-k:
k_11 = pm.loo(trace_m11_11, pointwise=True).pareto_k.values

In [ ]:
k_12 = pm.loo(trace_m12_2, pointwise=True).pareto_k.values

* Note no Pareto distribution warning in the Gamma-Poisson model

* The posterior predictive check below shows a much higher level of variation in the Gamma-Poisson model (as expected)

In [ ]:
# adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_12.ipynb

k_11 /= k_11.max()
psize_11 = 250 * k_11
k_12 /= k_12.max()
psize_12 = 250 * k_12

_, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 6))

# Pure possiong predictions:
pm.plot_hdi(P_seq, lam1_11, color="b", fill_kwargs={"alpha": 0.2}, ax=ax0)
ax0.plot(P_seq, lmu1_11, color="b", alpha=0.7, label="high contact mean")

pm.plot_hdi(P_seq, lam0_11, color="k", fill_kwargs={"alpha": 0.2}, ax=ax0)
ax0.plot(P_seq, lmu0_11, "--", color="k", alpha=0.7, label="low contact mean")

# display observed data:
index = c_id == 1
ax0.scatter(
    d_k.population[~index],d_k.total_tools[~index],s=psize_11[~index],
    facecolors="none",edgecolors="k",alpha=0.8,lw=1,label="low contact",
)
ax0.scatter(
    d_k.population[index], d_k.total_tools[index], s=psize_11[index], alpha=0.8, label="high contact"
)
plt.setp(ax0.get_xticklabels(), ha="right", rotation=45)
ax0.set_xlim((-10_000, 300_000))
ax0.set_xlabel("population")
ax0.set_ylabel("total tools")
ax0.set_ylim((-5, 125))
ax0.set_title("Pure Poisson model")
ax0.legend(fontsize=8, ncol=2)

# Gamma-Poisson predictions:
pm.plot_hdi(P_seq, lam1_12, color="b", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu1_12, color="b", alpha=0.7)

pm.plot_hdi(P_seq, lam0_12, color="k", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu0_12, "--", color="k", alpha=0.7)

# display observed data:
ax1.scatter(
    d_k.population[~index],d_k.total_tools[~index],s=psize_12[~index],
    facecolors="none",edgecolors="k",alpha=0.8,lw=1
)
ax1.scatter(
    d_k.population[index], d_k.total_tools[index], s=psize_12[index], alpha=0.8
)
plt.setp(ax1.get_xticklabels(), ha="right", rotation=45)
ax1.set_xlim((-10_000, 300_000))
ax1.set_xlabel("population")
ax1.set_ylim((-5, 125))
ax1.set_ylabel("total tools")
ax1.set_title("Gamma-Poisson model");

## Zero inflated outcomes

* Recorded data may not come from pure models

<br>

* Often, data come from multiple processes

<br>

* As we have seen above, mixture models allow us to capture several processes in one model

<br>

* In counting models, *zero* is particularly important
    * It might result from the processes not occurring
    * The process not progressing enough to produce anything
    * These different sources of zero outcomes produce **zero inflated models**

<br>

### Zero-Inflated Poisson: Writing Monastery

* We look at a zero-inflated Poisson model

<div style="float: right; margin-left: 20px">
    <img src="figs/Escribano.jpg" width=400px alt="https://en.wikipedia.org/wiki/File:Escribano.jpg">    
    <figcaption style="font-size: 8px"><a href="https://en.wikipedia.org/wiki/File:Escribano.jpg">Image source: Wikipedia</a></figcaption>
</div>

<br>

* Consider a monastery where monks produce a number of manuscripts every day. 

<br>

* We would like to predict the daily rate at which monks produce manuscripts
    * In other words, how many manuscripts monks produce per day on average

<br>

* Monks do not work every day, some days they they take breaks for wine tasting, those days they produce zero manuscripts

<br>

* However, it is also possible that monks produce zero manuscripts because, that day, the writing process went slowly.

<br>

* The process is illustrated in the figure below

<img src="figs/monks.png" width=300px>

<br>

* We use a zero-inflated Poisson model to estimate: i) the probability of the monks going to the wine cellar and ii) the rate of manuscripts they produce per day
    * We use $p$ to denote the probability that the monks go to the wine cellar
    * We use $\lambda$ to denote the rate of manuscript production

<br>
    
#### Likelihood for zero-inflated model

* To better understand the likelihood of this model, lets think of the data generating process as a mix of two processes:
    * With probability $p$ we select a process that always produces 0 manuscripts
    * With probability $1-p$ we select a process that generates manuscripts with rate $\lambda$

<br>

* Given the above, the likelihood of producing a zero manuscripts is given by

    \begin{align*}
    \mathrm{Pr}(0 \mid p,\lambda) = &\; \mathrm{Pr}(drink \mid p)\cdot \underbrace{1}_{\text{always no manuscripts when drinking}} + \mathrm{Pr}(work \mid p) \cdot \mathrm{Pr}(0 \mid \lambda) \\
    = &\; p + (1-p) \cdot \exp(-\lambda)
    \end{align*}

  where $\exp(-\lambda)$ is the Poisson pmf for outcome 0

<br>

* Similarly, the likelihood of producing non-zero manuscripts is given by

    \begin{align*}
    \mathrm{Pr}(y>0 \mid p,\lambda) = &\; \mathrm{Pr}(drink \mid p)\cdot \underbrace{0}_{\text{no manuscripts when drinking}} + \mathrm{Pr}(work \mid p) \cdot \mathrm{Pr}(y > 0 \mid \lambda) \\
    = &\; 0 + (1-p) \cdot \underbrace{\frac{\lambda^y\exp(-\lambda)}{y!}}_{\text{Poisson pmf}}
    \end{align*}

<br>

* This distribution exists and it is known as $\mathrm{ZIPoisson}$.

<br>

#### Data generating process

* We build a generative model for the data, and try to recover the value of the parameters using the ZIPoisson model

In [ ]:
# define parameters
PROB_DRINK = 0.2  # 20% of days
RATE_WORK = 1.0  # average 1 manuscript per day

# sample one year of production
N = 365

# simulate days monks drink
drink = np.random.binomial(1,PROB_DRINK, size=N)

# simulate manuscripts completed
y = (1 - drink) * np.random.poisson(RATE_WORK, size=N)

In [ ]:
zeros_drink = drink.sum()
zeros_work = (y == 0).sum() - zeros_drink

bins = np.arange(y.max() + 1) - 0.5
plt.hist(y, bins=bins, align="mid", rwidth=0.2, color="k", alpha=0.5, label="Working days")
plt.bar(0.0, zeros_drink, bottom=zeros_work, width=0.2, color="b", alpha=0.7, label="Drinking days")

plt.xticks(bins + 0.5)
plt.xlabel("manuscripts completed")
plt.legend()
plt.ylabel("Frequency");

#### Zero-Inflated Poisson model

* Note that in the PyMC function `ZeroINflatedPoission` we specify the probability of getting a 0
    * https://www.pymc.io/projects/docs/en/latest/api/distributions/generated/pymc.ZeroInflatedPoisson.html#pymc.ZeroInflatedPoisson
    * In our case study, this means the probability that the monks are on a working day (i.e., $1-p$)

In [ ]:
with pm.Model() as m12_3:
    l_λ = pm.Normal('l_λ',mu=1,sigma=.5)
    l_p = pm.Normal('l_p',mu=-1.5,sigma=1)
    λ   = pm.Deterministic('λ',pm.math.exp(l_λ))
    p   = pm.Deterministic('p',pm.math.invlogit(l_p))
    y   = pm.ZeroInflatedPoisson('y',1-p,λ,observed=y)

In [ ]:
pm.model_to_graphviz(m12_3)

In [ ]:
trace_m12_3 = pm.sample(5000,model=m12_3)

In [ ]:
pm.summary(trace_m12_3)

In [ ]:
pm.plot_posterior(trace_m12_3,var_names=['λ','p'],hdi_prob=.89,point_estimate='mode',ref_val=[RATE_WORK,PROB_DRINK]);

* The posterior of the parameters is very close to the original values
    * `λ` $\approx 1 =$ `RATE_WORK`
    * `p` $\approx 0.2 =$ `PROB_DRINK`

## Ordered Categorical Regression (Ordinal regression)

* Sometimes data is represented as a *discrete* indicative value
    * Positions in a race
    * Answers to questions in course evaluation surveys (*e.g.*, how much you agree with a statement)
    * This type of data is represented in an *ordinal scale*

<br>

* The models we have considered so far worked on a *metric* scale  
    * (except for multinomial regression, whose data is expressed in a *nominal* scale)

<br>

* The main difference with metric scales is that there is notion of distance between elements 
    * For instance, in the manuscript writing model, we can tell difference of manuscripts produced in two different days

<br>

* Consider the example of the positions in a race 
    * In an ordered scale, we can tell who finished first, second and so on
    * But we do not require that the distance between the first and the second is the same as second and third, or any other pair of positions.    
    * It is possible that the difference between 1 and 2 was 1 second, and between 2 and 3 5 seconds.

### Ordered categorical outcomes

* First we consider the case where the outcome variable is in ordinal scale

<br>

* In a nutshell, we re-use the standard multinomial model with the constraint that outcomes follow some order 
    * The model should ensure that, if a predictor is positively related to the ordered outcome, then larger values of the predictor increase the value in the ordinal scale
    * For instance, if age of the respondent is positively related to high values in course evaluations, the model must ensure that predictions increase as age increases

<br>

* The standard solution to the previous point is using a **cumulative link function**.  
    * Typically these are *cumulative probability/density functions*.
    * For instance, for a discrete distribution with positive support $X$, the cumulative probability of outcome 3 is $Pr(X \leq 3) = Pr(X=1) + Pr(X=2) + Pr(X=3)$
    * As another example, we show the cumulative density function of a normal distribution below
    
<br>

* In practice, a common cumulative link is the *cumulative-log-odds*

In [ ]:
X = np.linspace(-5,5,50)
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(7,3))
ax1.plot(X,sp.stats.norm.pdf(X));ax1.set_title('Normal pdf')
ax2.plot(X,sp.stats.norm.cdf(X));ax2.set_title('Normal cdf (cumulative)');

### Ethical Trolley

<figure>
  <img src="figs/trolley-ethics.png" width=500px alt="https://en.wikipedia.org/wiki/Trolley_problem#/media/File:Trolley_Problem.svg">
  <figcaption style="font-size: 10px;"><a href="https://en.wikipedia.org/wiki/Trolley_problem#/media/File:Trolley_Problem.svg">Image source: Wikipedia</a></figcaption>
</figure>





* We consider a dataset where repondents rate: How morally permissible is to pull the lever?
    * We use an ordinal scale from 1 (least permissible) to 7 (most permissible)
    
    

#### Data

In [ ]:
d_tr = pd.read_csv("Data/Trolley.csv", sep=";")
d_tr.head()

In [ ]:
d_tr.response.value_counts(sort=False).sort_index().plot(kind='bar');

In [ ]:
pr=d_tr.response.value_counts(normalize=True,sort=False).sort_index().cumsum()
pr.plot(marker='o',mfc='w')
plt.xlim(0.5, 7.5);

#### Cumulative-log-odds

* Turning the cumulative probabilities above into cumulative-log-odds defines an intercept for each outcome as expected

$$
\log \frac{\mathrm{Pr}(y_i \leq k)}{1-\mathrm{Pr}(y_i \leq k)} = \alpha_k
$$

* The concrete value for each intercept in the data is printed and plotted below

In [ ]:
d_tr.response.value_counts(normalize=True,sort=False).sort_index().cumsum().apply(sp.special.logit).values.round(2)

In [ ]:
d_tr.response.value_counts(normalize=True,sort=False).sort_index().cumsum().apply(sp.special.logit).plot(marker='o',mfc='w')
plt.xlim(0.5, 7.5);

<div style="width:400px;  height: 50px; display:flex; align-items:center; justify-content:center; background:blue; text-align:center; box-sizing:border-box; margin:8px auto; color: white">
    <strong style="font-size: 16px">
        Note that $\alpha_7 = \infty$. Why is this the case? <br> Is it a problem for the model we are building?
    </strong>
</div>

<br>

* In fact, we only need to know the probability of all outcomes except 1 ($K-1$)
    * The law of total probability states that $\sum_i \mathrm{Pr}(X=i) = 1 $
    * Knowing the probability of all outcomes except for one tells us the missing probability of the outcome

<br>
    
### Probability for each outcome

* So far we have discussed only cumulative probability, but we are actually interested in the probability for each outcome (not the cumulative)

* It is easy to use cumulative probabilities to compute the probability of each outcome

* Let $p_k$ denote the probability of outcome $k$

    $$
    p_k = \mathrm{Pr}(y_i \leq k) - \mathrm{Pr}(y_i \leq k - 1)
    $$

    That is, the probability of an outcome $k$ is the cumulative probability for $k$ minus the cumulative probability of $k-1$
  

* Now we show this graphically

In [ ]:
X = np.arange(1,8)
plt.plot(X,pr,marker='o',mfc='w')
plt.fill_between(X[:4],pr[X[:4]],alpha=.7,color='orange')
plt.fill_between(X[:3],pr[X[:3]],alpha=.7);
plt.title('Probability of outcome 4 (orange area)');

* Another version of the figure 12.5 in the textbook
    * I set the probability of outcomes as a bar from the bottom of the y-axis

In [ ]:
X = np.arange(1,8)
plt.plot(X,pr,marker='o',mfc='w', label='cumulative probability', color='blue')
plt.bar(X,pr,width=.1, label='outcome probability', color='orange')
outcome_pr = [0,
              pr[1],
              pr[2],
              pr[3],
              pr[4],
              pr[5],
              pr[6]]
plt.bar(X,outcome_pr,width=.1, label='cumulative probability outcome - 1', color='blue')
plt.legend();

### Ordered Regression for ethical trolley

* We consider the following model

  \begin{align*}
  R_i &\sim \mathrm{Categorical}(\vec{p}) \\
  p_1 &=  q_1 \\
  p_k &=  q_k - q_{k-1} \text{ where } k \in \{2,\ldots,K-1\} \\
  p_K &=  1 - q_{K-1} \\
  \textrm{logit}(q_k) &= \kappa_k - \varphi_i \\
  \varphi_i &= \text{terms of the linear model} \\
  \kappa_k &\sim \mathrm{Normal}(0,1.5)
  \end{align*}

  where $q_i$ is the cumulative-log-odds for each outcome,  $p_i$ the probability of each outcome, and $\vec{p} = \langle p_1, p_2, \ldots, p_K  \rangle$.

<br>

* This model is so common that there exists a tailored distribution, $\textrm{Ordered-logit}$, that encapsulate all the above

  \begin{align*}
  R_i &\sim \textrm{Ordered-logit}(\varphi_i,\kappa_k) \\
  \varphi_i &= \text{terms of the linear model} \\
  \kappa_k &\sim \mathrm{Normal}(0,1.5)
  \end{align*}


In [ ]:
np.arange(6) - 2.5

In [ ]:
with pm.Model() as m12_4:
    κ = pm.Normal('κ',mu=0,sigma=1.5,
                  transform=pm.distributions.transforms.ordered, # tells pymc that the vector of parameters must be ordered
                  shape=6,
                  initval=np.arange(6) - 2.5) # ensure that the initial values are ordered 
    φ = 0
    R = pm.OrderedLogistic('R',φ,κ,observed=d_tr.response.values-1) # 0 lowest, 6 maximum
    
    # cumulative probabilities in cumulative scale
    p = pm.Deterministic('p',pm.math.invlogit(κ))    

In [ ]:
pm.model_to_graphviz(m12_4)

In [ ]:
trace_m12_4 = pm.sample(model=m12_4)

In [ ]:
pm.summary(trace_m12_4,round_to=2)

In [ ]:
trace_m12_4.posterior.R_probs.mean(['chain','draw']).values

In [ ]:
# expected outcome
Ep = np.dot(trace_m12_4.posterior.R_probs.mean(['chain','draw']).values, np.arange(1,8))
Ep

In [ ]:
pm.plot_posterior(np.array([np.dot(x,np.arange(1,8)) for x in trace_m12_4.posterior.R_probs.mean(['chain']).values]));

### Adding predictor variables

* The model above is simply a Bayesian version of a histogram


* We can easily add predictors in the $\varphi_i$ parameter


* Here we will use a linear model

\begin{align*}
\log \frac{\mathrm{Pr}(y_i \leq k)}{1 - \mathrm{Pr}(y_i \leq k)} &= \alpha_k - \varphi_i \\
\varphi_i &= \beta x_i
\end{align*}


* Why minus (-) above? Because deduction values preserves the ordered scale
    
    * The expected outcome increases if we decrease the log-odds for all outcomes
    
    * Recall: $\mathbf{E}[y] = \sum_i p_i y_i$ where $y_i$ represent the value of outcome $i$ and $p_i$ is the probability the outcome.
    
    * The code below demonstrates this effect

In [ ]:
Ep

In [ ]:
cumprobs2 = logistic(trace_m12_4.posterior.κ.mean(['chain','draw']).values+0.5)
probs2 = [cumprobs2[0]] + [cumprobs2[i]-cumprobs2[i-1] for i in range(1,6)] + [1-cumprobs2[5]]
np.dot(probs2, np.arange(1,8))

#### Adding `action`, `intention` and `contact`

* The trolley dataset contains 3 indicator variables we will use as predictors

    * `action` - The actor performs an action that causes harm (e.g., pulls the lever)
      
    * `intention` - The actor causes harm intended as a means to a goal
      
    * `contact` - The actor uses physical contact to cause harm



* The predictors define 3 versions of the riddle aiming at testing their impact


* The model should consider the following possibilities:

    1. No action, contact, or intention
    
    2. Action
    
    3. Contact
    
    4. Intention
    
    5. Action and Intention
    
    6. Contact and intention

* To this end the cumulative-log-odds is defined as:

\begin{align*}
\log \frac{\mathrm{Pr}(y_i \leq k)}{1 - \mathrm{Pr}(y_i \leq k)} &= \alpha_k - \varphi_i \\
\varphi_i &= \beta_A A_i + \beta_C C_i + \mathrm{B}_{I,i}I_i \\
\mathrm{B}_{I,i} &= \beta_I + \beta_{IA} A_i + \beta_{IC}C_i
\end{align*}


<br>

<div style="width:400px;  height: 80px; display:flex; align-items:center; justify-content:center; background:blue; text-align:center; box-sizing:border-box; margin:8px auto; color: white">
    <strong style="font-size: 16px">
        Is this an interactive combination of predictors? <br> If so, can we map them to a treatment variable <br> (as in the chimpanzees model)?
    </strong>
</div>

In [ ]:
with pm.Model() as m12_5:
    α = pm.Normal('α',mu=0,sigma=1.5,
                  transform=pm.distributions.transforms.ordered, # tells pymc3 that this paramters are log-cumulative-oods of an ordinal scale
                  shape=6,
                  initval=np.arange(6) - 2.5) # ensure that the initial values are ordered 
    βA  = pm.Normal('βA',mu=0,sigma=0.5)
    βI  = pm.Normal('βI',mu=0,sigma=0.5)
    βC  = pm.Normal('βC',mu=0,sigma=0.5)
    βIA = pm.Normal('βIA',mu=0,sigma=0.5)
    βIC = pm.Normal('βIC',mu=0,sigma=0.5)
    
    # Predictors from data
    A = pm.Data("A", d_tr.action.values)
    I = pm.Data("I", d_tr.intention.values)
    C = pm.Data("C", d_tr.contact.values)        
    
    BI = βI + βIA*A + βIC*C
    φ  = pm.Deterministic('φ',βA*A + βC*C + BI*I)
    
    R  = pm.OrderedLogistic('R',φ,α,observed=d_tr.response.values-1) # 0 lowest, 6 maximum     

In [ ]:
pm.model_to_graphviz(m12_5)

In [ ]:
trace_m12_5 = pm.sample(model=m12_5)

In [ ]:
pm.summary(trace_m12_5,var_names=['βIC','βIA','βC','βI','βA'])

In [ ]:
pm.plot_forest(trace_m12_5,var_names=['βIC','βIA','βC','βI','βA'],combined=True,hdi_prob=.89,figsize=(5, 3));

### Posterior predictive check and posterior predictions

* It is somehow difficult to effectively plot the posterior predictive check or posterior predictions for ordinal models
    * The reason is that, each sample, is a vector of probabilities that depends on the values of the predictors
<br>

* To generate posterior predictions, a standard approach is to pick a predictor and plot the cumulative probability of all outcomes for all values of the predictor
    * This requires fixing the values of the other predictors if any

<br>

* Below we show two possibilities (a bit different from what it is shown in the book) for a posterior predictive check (1) and to generate posterior predictions (2)
    1. **Posterior predictive check**. We compare the posterior probability of each outcome (with the 95% HDI)－this is essentially a Bayesian version of a histogram, and the frequency of each answer in the data
    2. **Generating posterior predictions**. We use different predictor values and comparing the different in response probabilities
        * We consider fix different values of `action` and `contact` and compare the effect of `intention` 

In [ ]:
# data probability
prct_dict = d_tr.response.value_counts(normalize=True, ascending=True).to_dict()
sorted_prct = [prct_dict[k] for k in sorted(prct_dict)]

# posterior probability (mean and hdi)
means = trace_m12_5.posterior.R_probs.mean(dim=['chain','draw','R_probs_dim_0'])
hdis = pm.hdi(trace_m12_5.posterior.R_probs, hdi_prob=.95).mean(dim=['R_probs_dim_0']).R_probs

# Posterior predictive check using posterior distribution of probability
categories = np.arange(1,8)
plt.bar(categories, means, yerr=[means-hdis[:,0], hdis[:,1]-means], 
        capsize=4, color='skyblue', edgecolor='black', width=0.5, label='Posterior')
plt.scatter(categories, sorted_prct, color='red', marker='o', label='Data')
plt.xlabel('Response')
plt.ylabel('Probability')
plt.legend()
plt.show();

In [ ]:
# Posterior predictive analysis for probability (a bit different than McElreath's)
fig, axs = plt.subplots(1,3, figsize=(15,4))

for ((a,c),ax) in zip([(0,0), (1,0), (0,1)],axs):

    # posterior predictions
    with m12_5:
        ## no action, no contact, no interaction
        pm.set_data({'A': [a], 'C': [c], 'I': [0]})
        trace_m12_5_pp_0 = pm.sample_posterior_predictive(trace_m12_5, var_names=['R_probs'], progressbar=False)
        ## no action, no contact, interaction
        pm.set_data({'A': [a], 'C': [c], 'I': [1]})
        trace_m12_5_pp_1 = pm.sample_posterior_predictive(trace_m12_5, var_names=['R_probs'], progressbar=False)
    
    categories = np.arange(1,8)
    values1 = trace_m12_5_pp_0.posterior_predictive.R_probs.mean(dim=['chain','draw']).sel(R_probs_dim_0=0).values
    values2 = trace_m12_5_pp_1.posterior_predictive.R_probs.mean(dim=['chain','draw']).sel(R_probs_dim_0=0).values
    errors1 = [values1 - pm.hdi(trace_m12_5_pp_0.posterior_predictive.R_probs).R_probs.values[0,:,0],
               pm.hdi(trace_m12_5_pp_0.posterior_predictive.R_probs).R_probs.values[0,:,1]-values1]
    errors2 = [values2 - pm.hdi(trace_m12_5_pp_1.posterior_predictive.R_probs).R_probs.values[0,:,0],
               pm.hdi(trace_m12_5_pp_1.posterior_predictive.R_probs).R_probs.values[0,:,1]-values2]
    
    # Number of categories
    n = len(categories)
    
    # Positions of the bars on the x-axis
    ind = np.arange(n)
    
    # Width of the bars
    width = 0.35
    
    # Create bar plots    
    bar1 = ax.bar(ind - width/2, values1, width, yerr=errors1, 
                  capsize=4, label='Intention: 0', color='gray', edgecolor='black')
    bar2 = ax.bar(ind + width/2, values2, width, yerr=errors2, 
                  capsize=4, label='Intention: 1', color='cyan', edgecolor='black')
    
    ax.set_xlabel('Response')
    ax.set_ylabel('Probability')
    ax.set_title(f'Action: {a}, Contact: {c}')
    ax.set_xticks(ind)
    ax.set_xticklabels(categories)
    ax.legend()
    
# Show plot
plt.show()

### Ordered Predictor

* We add a predictor in the ordinal scale

* We use amount of education, `edu` in the `Trolley` dataset, as an ordinal predictor.

In [ ]:
d_tr

In [ ]:
d_tr["edu_new"] = pd.Categorical(
    d_tr.edu.values,
    categories=[
        "Elementary School", # lowest
        "Middle School",
        "Some High School",
        "High School Graduate",
        "Some College",
        "Bachelor's Degree",
        "Master's Degree",
        "Graduate Degree", # highest
    ],
    ordered=True,
)
d_tr["edu_new"] = d_tr.edu_new.cat.codes

* As for outcomes, the notion of ordered predictor implies an increment as you ascend in the ordinal scale

<br>

* We will use a parameter $\delta_i$ for each element in the ordinal scale

<br>

* To encode the cumulative nature of the scale we add in a linear fashion

\begin{align*}
\text{(Individual at level 1)}~~~~~~\varphi_i &= \delta_1 & +~\textit{other predictors in the linear model}  \\
\text{(Individual at level 2)}~~~~~~\varphi_i &= \delta_1 + \delta_2 & +~\textit{other predictors in the linear model} \\
& \ldots \\
\text{(Individual at level 7)}~~~~~~\varphi_i &= \delta_1 + \delta_2 + \ldots + \delta_7 & +~\textit{other predictors in the linear model} \\
\end{align*}

<br>

* The above can be compactly expressed as

  $$
  \varphi_i = \sum^{E_i - 1}_{j=0} \delta_j + \textit{other predictors in the linear model}
  $$
  
  where $E_i$ denotes the education level of the $i$th individual. 
     * Note that we have introduced a term $\delta_0$, this is a technicality to be able to write the model in a compact form. In fact, it is a constant $\delta_0=0$ as the parameter for the lowest ordinal value is the intercept.

<br>

* To complete the term, we multiply it by a coefficient $\beta_E$ which captures the effect of education on the outcome

<br>

* It is required that the sum of all terms equals $\sum_k \delta_k = 1$

<br>

* The model above is defined as follows to include the ordinal predictor:

  \begin{align*}
  R_i &\sim \textrm{Ordered\textrm-logit}(\varphi_i,\kappa_k) \\
  \varphi_i &= \underbrace{\beta_E \sum^{E_i - 1}_{j=0}\delta_j}_{\text{Ordinal predictor}} + \beta_A A_i + \beta_C C_i + \mathrm{B}_{I,i}I_i \\
  \mathrm{B}_{I,i} &= \beta_I + \beta_{IA} A_i + \beta_{IC}C_i \\
  \kappa_k &\sim \mathrm{Normal}(0,1.5) \\
  \beta_A, \beta_I, \beta_C, \beta_E &\sim \mathrm{Normal}(0,1) \\
  \delta &\sim \mathbf{Dirichlet(\alpha)} \\
  \end{align*}


#### Dirichlet distribution

* It is an instance of a _multivariate_ distribution
    * Probability distribution involving more than one variable

<br>

* It is a generalization of Beta    

<br>

* Takes as input a vector $\alpha$ of pseudo-counts for each variable
    * Each element of the vector indicates how often the variable in that position appears in comparison with the others

<br>

* The support is a vector of real numbers in $(0,1)$ that sum up to 1, i.e., a _simplex_ vector.
    * **This property is what makes it appropriate as a prior for $\delta$**

<br>

* See the bonus material on the Dirichlet distribution (directory `110-mixtures/dirichlet-bonus-material/`) for additional examples and an exercise

In [ ]:
with pm.Model() as m12_6:
    α = pm.Normal('α',mu=0,sigma=1.5,
                  transform=pm.distributions.transforms.ordered, # tells pymc that this paramters are log-cumulative-oods of an ordinal scale
                  shape=6,
                  initval=np.arange(6) - 2.5) # ensure that the initial values are ordered 
    βA  = pm.Normal('βA',mu=0,sigma=0.5)
    βI  = pm.Normal('βI',mu=0,sigma=0.5)
    βC  = pm.Normal('βC',mu=0,sigma=0.5)
    βIA = pm.Normal('βIA',mu=0,sigma=0.5)
    βIC = pm.Normal('βIC',mu=0,sigma=0.5)
    βE  = pm.Normal('βE',mu=0,sigma=0.5)
    
    δ   = pm.Dirichlet('δ',np.repeat(2.0,7),shape=7)
    δ_i = tt.concatenate([tt.zeros(1),δ])
    
    
    # Predictors from data
    A = pm.Data("A", d_tr.action.values)
    I = pm.Data("I", d_tr.intention.values)
    C = pm.Data("C", d_tr.contact.values)        
    E = pm.Data("E", d_tr.edu_new.values)
    
    BI = βI + βIA*A + βIC*C
    φ  = pm.Deterministic('φ',βE*(tt.cumsum(δ_i)[E]) + βA*A + βC*C + BI*I)
    
    R  = pm.OrderedLogistic('R',φ,α,observed=d_tr.response.values-1) # 0 lowest, 6 maximum   

In [ ]:
pm.model_to_graphviz(m12_6)

In [ ]:
trace_m12_6 = pm.sample(model=m12_6,target_accept=.9)

In [ ]:
pm.summary(trace_m12_6,var_names=['βIC','βIA','βC','βI','βA','βE','δ'],round_to=2)

* These results indicate Education has a _negative impact_ on the outcome (morality level)
    * The more educate the lower morality score the participants selects
 
<br>

* The values of $\delta_i$ parameters are negatively correlated due to the constraint that $\delta_i = 1$

<br>

* All education levels result in lower permissiveness, except for *Some College* whose values are very close to 0
    * See the posterior plots in the diagonal for this insight

In [ ]:
trace_m12_6.posterior.coords["δ_dim_0"] = [
    "Elem",
    "MidSch",
    "SHS",
    "HSG",
    "SCol",
    "Bach",
    "Mast",
]

pm.plot_pair(
    trace_m12_6,
    var_names=["δ"],
    marginals=True,    
    point_estimate="mean"
);